# Week 3: Prompts as Engineering Artifacts

Dependencies: `sentence-transformers` (local). For live calls: `pip install openai` and a Gemini key.

In [533]:
%pip install openai

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [534]:
from dotenv import load_dotenv
import os

load_dotenv()

key = os.environ.get("GEMINI_API_KEY")


In [535]:
import json
import hashlib
from pathlib import Path

CACHE_FILE = Path("response_cache.json")

if CACHE_FILE.exists():
    response_cache = json.loads(CACHE_FILE.read_text())
else:
    response_cache = {}

def make_cache_key(version, prompt, t):
    raw = f"{version}|{prompt}|{t['incident']}"
    return hashlib.sha256(raw.encode()).hexdigest()

In [536]:
import os, json, pathlib
import time
from openai import OpenAI, RateLimitError

def gemini_chat(messages, model="gemini-3.5-flash-lite", **kw):
    key = os.environ.get("GEMINI_API_KEY")
    if not key:
        return None

    client = OpenAI(
        api_key=key,
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
    )

    for attempt in range(3):
        try:
            return client.chat.completions.create(
                model=model,
                messages=messages,
                **kw
            ).choices[0].message.content

        except RateLimitError:
            if attempt == 2:
                raise

            wait = 2 ** attempt
            print(f"Rate limited. Retrying in {wait} seconds...")
            time.sleep(wait)
LIVE = os.environ.get('GEMINI_API_KEY') is not None
print('live model calls:', LIVE)

live model calls: True


In [537]:
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
from sentence_transformers import SentenceTransformer, util
emb = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
def exact_match(a, b):
    return float(str(a).strip().lower() == str(b).strip().lower())
def semantic_sim(a, b):
    e = emb.encode([a, b], convert_to_tensor=True, normalize_embeddings=True)
    return round(float(util.cos_sim(e[0], e[1])), 3)
print('metrics ready (exact-match + semantic)')

metrics ready (exact-match + semantic)


## Part 1: Versioned prompts and a test suite

Task: Classify software incidents into application_bug, infrastructure, configuration, database, or authentication. V2 adds a more specific decision rule for ambiguous incidents, directing the model to classify based on the underlying cause rather than the affected component.

In [538]:
from pathlib import Path

PROMPT_V1 = Path("prompts/incident_triage_v1.txt").read_text()
PROMPT_V2 = Path("prompts/incident_triage_v2.txt").read_text()
tests = [

  {'id':1,
   'incident':'The API returns 401 because the user access token expired.',
   'cat':'authentication',
   'why':'expired authentication token'},

  {'id':2,
   'incident':'The service fails to start because the DATABASE_URL environment variable is missing.',
   'cat':'configuration',
   'why':'missing environment configuration'},

  {'id':3,
   'incident':'The application throws a NullPointerException when the customer name is null.',
   'cat':'application_bug',
   'why':'application does not handle null value'},

  {'id':4,
   'incident':'The application cannot connect to PostgreSQL because the database server is unavailable.',
   'cat':'database',
   'why':'database service unavailable'},

  {'id':5,
   'incident':'The ECS task keeps restarting because the container fails its load balancer health check.',
   'cat':'infrastructure',
   'why':'container infrastructure health check failure'},

  {'id':6,
   'incident':'Users cannot log in because the configured OAuth client secret is invalid.',
   'cat':'authentication',
   'why':'invalid authentication credentials'},

  {'id':7,
   'incident':'The production application calls the test API because an environment variable contains the wrong URL.',
   'cat':'configuration',
   'why':'incorrect environment configuration'},

  {'id':8,
   'incident':'A report returns duplicate rows because the SQL query joins the same table incorrectly.',
   'cat':'database',
   'why':'incorrect database query'},

  {'id':9,
   'incident':'The checkout total is wrong because the application applies the discount twice.',
   'cat':'application_bug',
   'why':'incorrect application logic'},

  {'id':10,
   'incident':'The application cannot reach the database because a security group blocks the database port.',
   'cat':'infrastructure',
   'why':'network rule blocks database traffic'},

]
print('prompt versions:', 2, '| test cases:', len(tests))

prompt versions: 2 | test cases: 10


In [539]:
def run_case(version, prompt, t):
    cache_key = make_cache_key(version, prompt, t)

    if cache_key in response_cache:
        cached = response_cache[cache_key]
        return cached["category"], cached["rationale"]

    if not LIVE:
        raise RuntimeError(
            "GEMINI_API_KEY is required for uncached model calls."
        )

    txt = gemini_chat([
        {
            'role': 'user',
            'content': prompt + '\nIncident: ' + t['incident']
        }
    ])

    # Remove Markdown code fences if Gemini returns ```json ... ```
    txt = txt.strip()

    if txt.startswith("```json"):
        txt = txt.removeprefix("```json").removesuffix("```").strip()
    elif txt.startswith("```"):
        txt = txt.removeprefix("```").removesuffix("```").strip()

    try:
        d = json.loads(txt)
        category = d.get('category', '')
        rationale = d.get('rationale', '')
    except Exception:
        category = ''
        rationale = txt or ''

    response_cache[cache_key] = {
        "category": category,
        "rationale": rationale
    }

    CACHE_FILE.write_text(
        json.dumps(response_cache, indent=2)
    )

    return category, rationale

def score(version, prompt):
    rows = []
    for t in tests:
        cat, why = run_case(version, prompt, t)
        rows.append({'id':t['id'],'exact':exact_match(t['cat'],cat),'sem':semantic_sim(t['why'],why),'got':cat})
    acc = sum(r['exact'] for r in rows)/len(rows)
    return acc, rows

acc1, r1 = score('v1', PROMPT_V1)
acc2, r2 = score('v2', PROMPT_V2)

avg_sem_v1 = sum(r['sem'] for r in r1) / len(r1)
avg_sem_v2 = sum(r['sem'] for r in r2) / len(r2)


print(f"V1 exact match: {acc1:.0%}")
print(f"V2 exact match: {acc2:.0%}")

print(f"V1 average semantic similarity: {avg_sem_v1:.3f}")
print(f"V2 average semantic similarity: {avg_sem_v2:.3f}")

for t in tests:
    v1 = next(r for r in r1 if r['id'] == t['id'])
    v2 = next(r for r in r2 if r['id'] == t['id'])

    diff = v2['sem'] - v1['sem']

    if diff > 0:
        verdict = "IMPROVED"
    elif diff < 0:
        verdict = "REGRESSED"
    else:
        verdict = "SAME"

    print(
        f"Test {t['id']}: "
        f"V1={v1['sem']:.3f}, "
        f"V2={v2['sem']:.3f}"
    )



V1 exact match: 100%
V2 exact match: 90%
V1 average semantic similarity: 0.628
V2 average semantic similarity: 0.607
Test 1: V1=0.655, V2=0.672
Test 2: V1=0.814, V2=0.487
Test 3: V1=0.943, V2=0.943
Test 4: V1=0.569, V2=0.632
Test 5: V1=0.648, V2=0.612
Test 6: V1=0.628, V2=0.633
Test 7: V1=0.701, V2=0.664
Test 8: V1=0.587, V2=0.483
Test 9: V1=0.481, V2=0.418
Test 10: V1=0.251, V2=0.530


## Part 3 and 4: the tradeoff and the failure
Show one case the edit improved and one it regressed. The regression is your required failure.

In [540]:
for t in tests:
    e1 = next(r for r in r1 if r['id']==t['id'])['exact']
    e2 = next(r for r in r2 if r['id']==t['id'])['exact']
    if e1 != e2:
        verdict = 'IMPROVED' if e2 > e1 else 'REGRESSED'
        print(f"#{t['id']} expected {t['cat']!r}: v1 {'ok' if e1 else 'miss'} -> v2 {'ok' if e2 else 'miss'}  [{verdict}]")


#6 expected 'authentication': v1 ok -> v2 miss  [REGRESSED]


## Prompt Evaluation

I evaluated two versions of a software incident triage prompt. Both use few-shot examples, chain-of-thought prompting, and reasoning guidance. V2 adds a more specific rule for ambiguous incidents involving multiple components.

A test case that regressed was Test 6, which evaluated whether the model could correctly classify an authentication error. V1 correctly classified the incident, while V2 incorrectly classified it. This represents an exact-match regression. Interestingly, the semantic similarity of the rationale slightly improved from 0.628 in V1 to 0.633 in V2. The edit regressed on this input as my additional chain of prompt text added additional clarification around multi-component errors which may have caused the model to overanalyze the prompt and focus on secondary issues rather than the primary cause. This can be resolved by making the categorical boundaries more explicit.

A test case that improved was Test 4, which evaluated whether the model could identify an incident as a database issue. Both prompts correctly classified the incident based on exact match. The semantic similarity of the rationale improved from 0.569 in V1 to 0.632 in V2. This increase in semantic similarity can be attributed to additional guidance in V2 for the model to consider the underlying cause.


## Part 5: Submit
Store the prompt versions as files, run the suite (set your key for real calls), and open a pull request with the metric numbers and a linked research note. Rubric: versioned prompts (15), structured prompt (20), test suite with two metrics (25), tradeoff with numbers (25), PR hygiene (15).